# Sparse Readout Prism — demo

Trains a tiny readout-feature dictionary on a self-contained **synthetic** readout
(no model download, no GPU, ~1 s on CPU), decomposes a single token logit into
`base + sparse features + residual` as an exact identity, then ranks the
contributing features and looks up the labels the pipeline assigned them.

For decomposing a real LM's logits, see the "Decomposing a real model's logits"
section of the [README](../README.md).

## Setup

Running locally from a clone, install the package once with `uv sync` (or
`pip install -e .`) and run this notebook with that environment. On Colab,
uncomment the cell below.

In [ ]:
import os

# This notebook expects to run from the repo root; hop up if launched from notebooks/.
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

# Colab only — fetch and install the repo first (then the cell above is a no-op):
# !git clone https://github.com/hematteo/sparse-readout-prism.git
# %cd sparse-readout-prism
# !pip install -q -e .

## 1. Train a dictionary on the synthetic smoke config

Writes `checkpoint.pt` (+ config/metrics/history/feature_labels) under
`results/smoke_demo/`.

In [ ]:
import subprocess
import sys

# Use the kernel's own interpreter so this works under Colab, Jupyter, or uv.
subprocess.run(
    [
        sys.executable,
        "scripts/train/train_readout_sae_from_config.py",
        "--config",
        "configs/smoke.yaml",
        "--out-dir",
        "results/smoke_demo",
    ],
    check=True,
)

## 2. Decompose a token logit against the trained dictionary

The decomposition is an exact identity: `original_logit == base_term +
feature_sum + residual_term`.

In [ ]:
from sparse_readout_prism import decompose_token_logit, load_factorizer, preprocess_rows
from sparse_readout_prism.data import make_synthetic_data

# The smoke run trained on this synthetic readout (seed 7, d_model=128);
# regenerate it so the shapes match the checkpoint.
W_U, hidden = make_synthetic_data(seed=7)  # W_U: (vocab=1024, d_model=128)
h = hidden.reshape(-1, W_U.shape[1])[0]  # one final-norm hidden state
row_mean, row_norms, rows_normalized = preprocess_rows(W_U)

sae = load_factorizer("results/smoke_demo/checkpoint.pt", freeze=True)

token_id = int((h @ W_U.T).argmax())  # the token this h reads out as
d = decompose_token_logit(
    h=h,
    W_row=W_U[token_id],
    row_mean=row_mean,
    row_norm=row_norms[token_id],
    row_normalized=rows_normalized[token_id],
    model=sae,
    k=sae.k,
)

print(f"{d.original_logit:.3f} = {d.base_term:.3f} + {d.feature_sum:.3f} + {d.residual_term:.3f}")
print(f"active features: {d.active_feature_indices.tolist()}")

## 3. Inspect: identity check, top features, labels

`identity_error` is the bookkeeping residual of the exact identity — float
noise by construction. `feature_contributions` is indexed by feature id, so
slicing it with `active_feature_indices` ranks the features that actually
drove this logit; the smoke pipeline already labelled each sufficiently-used
feature by its top co-activating tokens, so we can name them.

In [ ]:
import json
from pathlib import Path

import torch

assert torch.abs(d.identity_error) < 1e-4

# Rank this token's active features by signed contribution, with their labels.
label_by_id = {
    f["feature_id"]: f["label"]
    for f in json.loads(Path("results/smoke_demo/feature_labels.json").read_text())["features"]
}
contrib = d.feature_contributions[d.active_feature_indices]
for rank in contrib.abs().argsort(descending=True)[:5]:
    fid = int(d.active_feature_indices[rank])
    print(f"feature {fid:4d}  {contrib[rank]:+8.3f}  {label_by_id.get(fid, '(below labelling usage threshold)')}")

# Dictionary-level quality metrics the same run wrote (synthetic toy numbers).
metrics = json.loads(Path("results/smoke_demo/metrics.json").read_text())
print({k: round(metrics[k], 4) for k in ("val_top1_match", "val_logit_kl_bits_mean", "dead_feature_rate")})